# Experimento Q-Learning - Parte 2

Aqui são executados os testes de acurácia.

Carrega-se os tabuleiros e tabelas Q e a memória somática da faze anterior, para cada dimensão 

Altera-se o tabuleiro de maneira a inserir um buraco no caminho do Agente, segundo a tabela Q.

Resolve-se o tabuleiro 100 vezes com um Agente Clássico e 100 vezes com o Agente Q-Learning Somático. 


In [1]:
from typing import NamedTuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm
import pickle

import gymnasium as gym
from gymnasium.envs.toy_text.frozen_lake import generate_random_map

from Agent import *

from Util import *


sns.set_theme()

In [2]:
#somatic parameters
la = 0.6
ld = -0.6
d = 0.8
f = 1

In [3]:
def run_testing(total_episodes, agent:Agent):

    rewards = np.zeros((total_episodes))
    steps = np.zeros((total_episodes))
    episodes = np.arange(total_episodes)

    all_states = []
    all_actions = []

    for episode in tqdm(
        episodes, desc=f"Testing Episodes", leave=False
        ):
        agent.setExecutionMode()
        agent.reset()        
        step = 0
        done = False
        total_rewards = 0

        while not done:
            state, action, _, reward = agent.step()

            # Log all states and actions
            all_states.append(state)
            all_actions.append(action)

            done = agent.done()

            total_rewards += reward
            step += 1

        # Log all rewards and steps
        rewards[episode] = total_rewards
        steps[episode] = step

    return rewards, steps, episodes, all_states, all_actions

In [52]:
#(size, hole(y,x))
map_sizes =[(8,(5,5)), (10,(4,5)), (12,(7,6))]

In [55]:

for map_size, hole in map_sizes:
    with open(f"maps/{map_size}x{map_size}.pkl", 'rb') as inf: 
        map = pickle.load(inf)

    
    map[int(hole[0])][int(hole[1])] = 'H'

    print("MAP SIZE {} ##############################################".format(map_size))

    env = gym.make(
        "FrozenLake-v1",
        is_slippery=False,
        render_mode="rgb_array",
        desc=map,
    )

    with open(f"data/qtable{map_size}.pkl", 'rb') as inf: 
        qtable = pickle.load(inf)

    agent = QLearning(env)

    print("CLASSIC Q-LEARNING ##############################################")

    agent.setQTable(qtable)

    rewards, steps, _,_,_ = run_testing(100,agent)

    print("Total de sucesso em 100:", rewards.sum())


    print("SOMATIC Q-LEARNING ##############################################")
    agent = SomaticQLearning(env,la,ld,d,f)

    with open(f"data/qtable{map_size}_s.pkl", 'rb') as inf: 
        qtable = pickle.load(inf)

    with open(f"data/sm{map_size}.pkl", 'rb') as inf: 
        sm = pickle.load(inf)
        
    agent.set_somaticMemory(sm)
    agent.setQTable(qtable)

    rewards, steps, _,_,_ = run_testing(100,agent)

    print("Total de sucesso em 100:", rewards.sum())

    print("\n\n\n")

MAP SIZE 8 ##############################################
CLASSIC Q-LEARNING ##############################################


Total de sucesso em 100: 0.0
SOMATIC Q-LEARNING ##############################################


Total de sucesso em 100: 99.0




MAP SIZE 10 ##############################################
CLASSIC Q-LEARNING ##############################################


Total de sucesso em 100: 0.0
SOMATIC Q-LEARNING ##############################################


Total de sucesso em 100: 99.0




MAP SIZE 12 ##############################################
CLASSIC Q-LEARNING ##############################################


Total de sucesso em 100: 51.0
SOMATIC Q-LEARNING ##############################################


Total de sucesso em 100: 98.0




